# 06 Experiment 3: Anomaly Detection

This notebook explores anomaly detection in review text using two distinct approaches:
1. **Structural Anomalies (Isolation Forest)**: Identifying reviews whose embeddings lie far from the general distribution of reviews. These might be spam, repetitive junk, or highly unusual text.
2. **Rating-Text Inconsistency (Regression)**: Training a model to predict the star rating from the embeddings, and flagging reviews with the largest errors. These are often sarcastic reviews or cases where the user accidentally clicked 1-star for a glowing review.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.append('..')

from src.anomaly import detect_outliers_isolation_forest, detect_rating_inconsistencies

## 1. Load Data

In [ ]:
processed_path = '../data/processed/cleaned_reviews.parquet'
df = pd.read_parquet(processed_path)
ratings = df['rating'].values
texts = df['cleaned_text'].values

print(f"Loaded {len(df)} reviews.")

## 2. Evaluate Anomaly Detection Methods

In [ ]:
embedding_dir = '../data/embeddings/'
results_dir = '../experiments/anomaly/'
figures_dir = '../experiments/figures/'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

# For this notebook, we'll just demonstrate on BGE embeddings since they are the richest.
# You can loop over 'encoders' just like in the previous notebooks to compare them all.
encoder_to_test = 'bge'
emb_path = os.path.join(embedding_dir, f'{encoder_to_test}.npy')

if not os.path.exists(emb_path):
    raise FileNotFoundError(f"{emb_path} not found. Run 03_encode.ipynb first.")
    
X = np.load(emb_path)
print(f"Loaded {encoder_to_test.upper()} embeddings with shape {X.shape}")

### Approach A: Isolation Forest (Top 1% Structural Outliers)

In [ ]:
is_iso_anomaly, iso_scores = detect_outliers_isolation_forest(X, contamination=0.01)
df['iso_anomaly'] = is_iso_anomaly
df['iso_score'] = iso_scores

# Plot the distribution of anomaly scores
plt.figure(figsize=(8, 4))
sns.histplot(iso_scores, bins=50, kde=True)
plt.title('Isolation Forest Anomaly Scores (Lower is more anomalous)')
plt.savefig(os.path.join(figures_dir, 'iso_forest_scores.png'))
plt.show()

In [ ]:
print("Top 5 Most Anomalous Reviews (Isolation Forest):\n")
iso_anomalies = df[df['iso_anomaly']].sort_values('iso_score').head(5)

for i, row in iso_anomalies.iterrows():
    print(f"Rating: {row['rating']}")
    print(f"Score:  {row['iso_score']:.3f}")
    print(f"Text:   {row['cleaned_text']}\n")
    print("-" * 50)

### Approach B: Rating-Text Inconsistency (Top 1% Residuals)

In [ ]:
is_inconsistent, residuals, pred_ratings = detect_rating_inconsistencies(X, ratings, top_percent=0.01)
df['inconsistent'] = is_inconsistent
df['rating_residual'] = residuals
df['predicted_rating'] = pred_ratings

# Plot the distribution of residuals
plt.figure(figsize=(8, 4))
sns.histplot(residuals, bins=50, kde=True)
plt.title('Rating Prediction Absolute Residuals (Higher means more inconsistent)')
plt.savefig(os.path.join(figures_dir, 'rating_inconsistency_residuals.png'))
plt.show()

In [ ]:
print("Top 5 Most Inconsistent Reviews (Text vs Rating):\n")
inconsistencies = df[df['inconsistent']].sort_values('rating_residual', ascending=False).head(5)

for i, row in inconsistencies.iterrows():
    print(f"Actual Rating: {row['rating']}")
    print(f"Pred Rating:   {row['predicted_rating']:.2f}")
    print(f"Residual:      {row['rating_residual']:.2f}")
    print(f"Text:          {row['cleaned_text']}\n")
    print("-" * 50)

## 3. Save Results

In [ ]:
# Save the flagged anomalies to CSV for manual inspection
df[df['iso_anomaly']].to_csv(os.path.join(results_dir, f'{encoder_to_test}_iso_anomalies.csv'), index=False)
df[df['inconsistent']].to_csv(os.path.join(results_dir, f'{encoder_to_test}_inconsistencies.csv'), index=False)

print(f"Saved anomaly CSVs to {results_dir}")